# RQ1 — Notebook 1: Dataset Pull

**Research Question**: Which clustering methods generate the best pseudo-labels for downstream classification?

This notebook builds the meta-training dataset pool from OpenML. It filters, downloads,
and verifies all datasets used in the benchmark.

**Key filters**:
- 100 ≤ n_instances ≤ 100,000
- **2 ≤ n_classes ≤ 10** (binary included — clustering is meaningful for 2+ classes)
- n_features < 200
- No missing values (after loading)
- Not a showcase dataset
- Not in the skip list (datasets too large for clustering to complete in reasonable time)

**Output**: `data/meta_table/dataset_manifest.csv`

In [7]:
import os, math, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml

ROOT     = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR  = os.path.join(ROOT, 'data', 'raw')
META_DIR = os.path.join(ROOT, 'data', 'meta_table')
MANIFEST = os.path.join(META_DIR, 'dataset_manifest.csv')

os.makedirs(RAW_DIR,  exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

openml.config.cache_directory = RAW_DIR

print(f'ROOT     : {ROOT}')
print(f'MANIFEST : {MANIFEST}')

ROOT     : c:\MLResearch
MANIFEST : c:\MLResearch\data\meta_table\dataset_manifest.csv


In [8]:
# Showcase datasets: held-out generalization test — NEVER in meta-training
SHOWCASE_IDS = {
    61,    # iris
    187,   # wine
    15,    # breast-w
    53,    # heart-statlog
    40966, # penguins
    37,    # diabetes
    54,    # vehicle
    1590,  # adult
    1597,  # creditcard
}

# Skip IDs: datasets confirmed too large for clustering to complete in reasonable time.
# Excluded at manifest level so they are never downloaded.
SKIP_IDS = {40685, 46536, 41166, 255, 119, 46955, 45548, 45927}

# All IDs to exclude from meta-training
EXCLUDE_IDS = SHOWCASE_IDS | SKIP_IDS

print(f'Showcase IDs  : {sorted(SHOWCASE_IDS)}')
print(f'Skip IDs      : {sorted(SKIP_IDS)}')
print(f'Total excluded: {len(EXCLUDE_IDS)}')

Showcase IDs  : [15, 37, 53, 54, 61, 187, 1590, 1597, 40966]
Skip IDs      : [119, 255, 40685, 41166, 45548, 45927, 46536, 46955]
Total excluded: 17


In [9]:
# Filter constants
MIN_INSTANCES = 100
MAX_INSTANCES = 100_000
MIN_CLASSES   = 2       # binary datasets included — clustering is valid for 2+ classes
MAX_CLASSES   = 10
MAX_FEATURES  = 200
TARGET_COUNT  = 120     # pool size before verification

print('Filters:')
print(f'  instances : {MIN_INSTANCES:,} – {MAX_INSTANCES:,}')
print(f'  classes   : {MIN_CLASSES} – {MAX_CLASSES}')
print(f'  features  : < {MAX_FEATURES}')
print(f'  missing   : none allowed')
print(f'  target    : {TARGET_COUNT} datasets after stratified sampling')

Filters:
  instances : 100 – 100,000
  classes   : 2 – 10
  features  : < 200
  missing   : none allowed
  target    : 120 datasets after stratified sampling


In [10]:
print('Fetching task list from OpenML (~30 s on first run)...')

tasks = openml.tasks.list_tasks(
    task_type=openml.tasks.TaskType.SUPERVISED_CLASSIFICATION,
    output_format='dataframe',
)

print(f'Total tasks returned: {len(tasks):,}')

Fetching task list from OpenML (~30 s on first run)...
Total tasks returned: 5,581


In [11]:
df = tasks.copy()

needed = ['NumberOfInstances', 'NumberOfFeatures', 'NumberOfClasses',
          'NumberOfMissingValues', 'did']
df = df.dropna(subset=needed)
for col in needed[:-1]:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=needed)

before = len(df)
df = df[
    (df['NumberOfInstances']     >= MIN_INSTANCES) &
    (df['NumberOfInstances']     <= MAX_INSTANCES) &
    (df['NumberOfClasses']       >= MIN_CLASSES)   &
    (df['NumberOfClasses']       <= MAX_CLASSES)   &
    (df['NumberOfFeatures']      <  MAX_FEATURES)  &
    (df['NumberOfMissingValues'] == 0)
]
print(f'After size/quality filters: {len(df):,}  (from {before:,})')

df = df.drop_duplicates(subset='did')
df = df[~df['did'].isin(EXCLUDE_IDS)]
df = df.sort_values('NumberOfInstances').reset_index(drop=True)
print(f'After dedup + exclusions: {len(df):,} candidate datasets')

After size/quality filters: 2,748  (from 5,520)
After dedup + exclusions: 1,304 candidate datasets


In [12]:
df['log_n'] = np.log10(df['NumberOfInstances'])

# balance_bin: use MinorityClassSize/MajorityClassSize when available, else 0
if 'MinorityClassSize' in df.columns and 'MajorityClassSize' in df.columns:
    ratio = (df['MinorityClassSize'] / df['MajorityClassSize']).clip(0, 1)
    df['balance_bin'] = ratio.fillna(0).gt(0.5).astype(int)
else:
    df['balance_bin'] = 0

df['class_bin'] = (df['NumberOfClasses'] >= 6).astype(int)
# labels=False returns integer bin codes; fillna(0) handles edge values outside bins
df['size_bin']  = pd.cut(df['log_n'], bins=3, labels=False).fillna(0).astype(int)
df['stratum']   = (df['class_bin'].astype(str) + '_'
                 + df['size_bin'].astype(str)  + '_'
                 + df['balance_bin'].astype(str))

strata_counts = df.groupby('stratum').size().sort_index()
print('Strata (class_bin _ size_bin _ balance_bin):')
print('  class_bin  : 0 = 2-5 classes,  1 = 6-10 classes')
print('  size_bin   : 0/1/2 = small/medium/large')
print('  balance_bin: 0 = imbalanced or unknown,  1 = balanced')
print()
print(strata_counts.to_string())
print(f'\nTotal available: {len(df):,}')

non_empty = strata_counts[strata_counts > 0]
per_strat = math.ceil(TARGET_COUNT / len(non_empty))

# Explicit loop + concat avoids pandas 2.x groupby/apply index-promotion behaviour
# that causes reset_index(drop=True) to silently drop the stratum column
frames = []
for stratum_val, group in df.groupby('stratum'):
    frames.append(group.sample(min(per_strat, len(group)), random_state=42))

sampled = (pd.concat(frames)
             .sample(frac=1, random_state=42)
             .head(TARGET_COUNT)
             .reset_index(drop=True))

print(f'\nSelected {len(sampled)} datasets')
print(sampled['stratum'].value_counts().sort_index().to_string())

Strata (class_bin _ size_bin _ balance_bin):
  class_bin  : 0 = 2-5 classes,  1 = 6-10 classes
  size_bin   : 0/1/2 = small/medium/large
  balance_bin: 0 = imbalanced or unknown,  1 = balanced

stratum
0_0_0    132
0_0_1    206
0_1_0    228
0_1_1    383
0_2_0     72
0_2_1    151
1_0_0     11
1_0_1      6
1_1_0     68
1_1_1     37
1_2_0      5
1_2_1      5

Total available: 1,304

Selected 106 datasets
stratum
0_0_0    10
0_0_1    10
0_1_0    10
0_1_1    10
0_2_0    10
0_2_1    10
1_0_0    10
1_0_1     6
1_1_0    10
1_1_1    10
1_2_0     5
1_2_1     5


In [13]:
records = []
failed  = []

for i, row in sampled.iterrows():
    did  = int(row['did'])
    name = row.get('name', '')
    try:
        ds = openml.datasets.get_dataset(
            did, download_data=True,
            download_qualities=True,
            download_features_meta_data=False,
        )
        X, y, _, _ = ds.get_data(
            dataset_format='dataframe',
            target=ds.default_target_attribute,
        )
        if X.isnull().any().any() or (y is not None and y.isnull().any()):
            failed.append((did, name, 'missing values after load'))
            continue

        n_inst    = X.shape[0]
        n_feat    = X.shape[1]
        n_classes = int(y.nunique()) if y is not None else 0

        records.append({
            'dataset_id' : did,
            'name'       : ds.name,
            'n_instances': n_inst,
            'n_features' : n_feat,
            'n_classes'  : n_classes,
        })
        print(f'[{i+1:3d}/{len(sampled)}] OK  id={did:6d}  {ds.name[:40]:40s}  '
              f'{n_inst:6d}r × {n_feat:3d}c  {n_classes}cls')

    except Exception as e:
        failed.append((did, name, str(e)))
        print(f'[{i+1:3d}/{len(sampled)}] FAIL id={did}  {name}  — {e}')

print(f'\nLoaded: {len(records)}  Failed: {len(failed)}')

[  1/106] OK  id= 46652  news_channel                               20284r ×  17c  6cls
[  2/106] OK  id=   876  fri_c1_100_50                                100r ×  50c  2cls
[  3/106] OK  id=   767  analcatdata_apnea1                           475r ×   3c  2cls
[  4/106] OK  id= 44521  fabert_seed_3_nrows_2000_nclasses_10_nco    2000r × 100c  7cls
[  5/106] OK  id= 45714  PriceRunner                                35300r ×   5c  10cls
[  6/106] OK  id=   285  flags                                        194r ×  28c  8cls
[  7/106] OK  id= 44406  bank-marketing_seed_4_nrows_2000_nclasse    2000r ×   7c  2cls
[  8/106] OK  id=     3  kr-vs-kp                                    3196r ×  36c  2cls
[  9/106] OK  id= 44627  mfeat-factors_seed_4_nrows_2000_nclasses    2000r × 100c  10cls
[ 10/106] OK  id=   775  fri_c2_100_25                                100r ×  25c  2cls
[ 11/106] OK  id=  1537  volcanoes-c1                               28626r ×   3c  5cls
[ 12/106] OK  id= 43923  mushr

In [14]:
# Attempt to replace any failed downloads from a candidate list
REPLACEMENT_CANDIDATES = [28, 29, 31, 36, 38, 44, 46, 50, 57, 60]

existing_ids = {r['dataset_id'] for r in records}

for did, name, reason in failed:
    replaced = False
    for cand_id in REPLACEMENT_CANDIDATES:
        if cand_id in existing_ids or cand_id in EXCLUDE_IDS:
            continue
        try:
            ds = openml.datasets.get_dataset(
                cand_id, download_data=True,
                download_qualities=True,
                download_features_meta_data=False,
            )
            X, y, _, _ = ds.get_data(
                dataset_format='dataframe',
                target=ds.default_target_attribute,
            )
            if X.isnull().any().any() or (y is not None and y.isnull().any()):
                continue
            n_inst, n_feat = X.shape
            n_cls = int(y.nunique()) if y is not None else 0
            if not (MIN_INSTANCES <= n_inst <= MAX_INSTANCES
                    and MIN_CLASSES <= n_cls <= MAX_CLASSES
                    and n_feat < MAX_FEATURES):
                continue
            rec = {'dataset_id': cand_id, 'name': ds.name,
                   'n_instances': n_inst, 'n_features': n_feat, 'n_classes': n_cls}
            records.append(rec)
            existing_ids.add(cand_id)
            print(f'Replaced failed id={did} with id={cand_id} ({ds.name})')
            replaced = True
            break
        except Exception:
            pass
    if not replaced:
        print(f'Could not replace failed id={did} ({name})')

In [15]:
manifest = pd.DataFrame(records)
manifest.to_csv(MANIFEST, index=False)

print(f'Manifest saved → {MANIFEST}')
print(f'Shape: {manifest.shape}')
print()

# Sanity checks
leaked = set(manifest['dataset_id']) & EXCLUDE_IDS
assert len(leaked) == 0, f'EXCLUSION LEAK: {leaked}'
assert manifest['n_instances'].between(MIN_INSTANCES, MAX_INSTANCES).all()
assert manifest['n_classes'].between(MIN_CLASSES, MAX_CLASSES).all()
assert (manifest['n_features'] < MAX_FEATURES).all()
assert manifest['dataset_id'].is_unique

print('All sanity checks passed.')
print()
print('=== Class distribution ===')
print(manifest['n_classes'].value_counts().sort_index())
print()
print('=== Dataset summary ===')
print(manifest[['n_instances', 'n_features', 'n_classes']].describe().round(1).to_string())
print(f'\nReady for 02_lse_computation.ipynb')

Manifest saved → c:\MLResearch\data\meta_table\dataset_manifest.csv
Shape: (106, 5)

All sanity checks passed.

=== Class distribution ===
n_classes
2     50
3      4
4      2
5      4
6     18
7      6
8      4
9      3
10    15
Name: count, dtype: int64

=== Dataset summary ===
       n_instances  n_features  n_classes
count        106.0       106.0      106.0
mean        8858.9        28.8        4.7
std        14594.0        32.3        3.0
min          100.0         3.0        2.0
25%          582.0         9.0        2.0
50%         2000.0        15.5        3.0
75%        10207.5        36.8        7.0
max        72998.0       129.0       10.0

Ready for 02_lse_computation.ipynb
